# Unidad 43. Matrices y mapas de calor

**Bloque:** Seaborn (UT3) · **Tiempo orientativo:** 5–7 horas

> Un mapa de calor codifica una tabla numérica con color. Solo es útil cuando conocemos el significado de cada fila, columna y escala; el color no sustituye a los valores ni convierte asociación en causalidad.

## Resultados de aprendizaje

Al terminar podrás crear e interpretar `heatmap` y `clustermap`; construir matrices de correlación y tablas pivote; aplicar una máscara triangular; y representar una matriz de confusión precomputada con etiquetas correctas.

**Prerequisitos:** Pandas (unidades 18–23), NumPy (25–29) y unidades 39–42 de Seaborn.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="white", context="notebook")


def ruta_datos(nombre):
    for ruta in [Path("datos") / "seaborn" / nombre, Path("..") / "datos" / "seaborn" / nombre]:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(f"No se encontró {nombre}. Abre el cuaderno desde la carpeta del curso.")

sensores = pd.read_csv(ruta_datos("43_medidas_sensores.csv"))
predicciones = pd.read_csv(ruta_datos("43_predicciones_clases.csv"))
sensores.head()

## 1. Correlación: una tabla antes que un color

`corr()` calcula asociaciones lineales entre columnas numéricas. El resultado es una matriz simétrica: la diagonal vale 1 porque cada variable se compara consigo misma. Una correlación alta no implica que una variable produzca la otra; además puede ocultar relaciones no lineales, grupos o factores no observados.

Antes de dibujar, muestra la tabla y confirma que solo contiene medidas comparables.

In [ ]:
columnas_numericas = ["temperatura_c", "humedad_pct", "co2_ppm", "pm25_ug_m3"]
correlaciones = sensores[columnas_numericas].corr()
correlaciones.round(2)

## 2. `heatmap`: escala, anotaciones y máscara

Un mapa de calor necesita una barra de color y una escala adecuada. Para correlaciones, `vmin=-1`, `vmax=1` y una paleta divergente sitúan el cero en un punto visual neutro. Las anotaciones permiten comprobar valores concretos sin depender solo del color.

Como la matriz es simétrica, mostrar los dos triángulos duplica información. Una máscara puede ocultar uno de ellos; no modifica los datos, solo la representación.

In [ ]:
mascara = np.triu(np.ones_like(correlaciones, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(correlaciones, mask=mascara, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, center=0, square=True, linewidths=0.5, cbar_kws={"label": "Correlación lineal"}, ax=ax)
ax.set(title="Correlaciones en lecturas simuladas")
plt.show()

## 3. Tablas pivote: resumir antes de representar

Una tabla pivote transforma filas en una matriz. Aquí calculamos la media de PM2.5 para cada zona y cada intervalo de temperatura. La agregación debe estar escrita en el código y en el título: un mapa de calor de medias no representa observaciones individuales.

Con pocas filas, dividir demasiado los datos puede dejar celdas vacías o resúmenes poco estables. Revisa siempre la tabla pivote antes de interpretar sus colores.

In [ ]:
sensores = sensores.assign(intervalo_temp=pd.cut(sensores["temperatura_c"], bins=[17, 19, 21, 23], include_lowest=True))
pivote_pm25 = sensores.pivot_table(index="zona", columns="intervalo_temp", values="pm25_ug_m3", aggfunc="mean", observed=False)
display(pivote_pm25)

fig, ax = plt.subplots(figsize=(8, 3))
sns.heatmap(pivote_pm25, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "PM2.5 medio (µg/m³)"}, ax=ax)
ax.set(title="Media simulada de PM2.5 por zona e intervalo", xlabel="Intervalo de temperatura (°C)", ylabel="Zona")
plt.show()

## 4. `clustermap`: ordenar para explorar, no para confirmar

`clustermap` reordena filas y columnas según su parecido. Puede revelar bloques visuales, pero el agrupamiento depende de la escala, la distancia y los datos disponibles. No es una prueba de que existan grupos reales. Seaborn crea una figura completa, por lo que no se pasa `ax`.

En este ejemplo usamos la matriz de correlaciones; no es necesario interpretar el dendrograma para completar la unidad.

In [ ]:
agrupado = sns.clustermap(correlaciones, cmap="vlag", vmin=-1, vmax=1, center=0, annot=True, fmt=".2f", figsize=(7, 7))
agrupado.figure.suptitle("Ordenación exploratoria de correlaciones", y=1.03)
plt.show()

## 5. Matriz de confusión: filas, columnas y clases

Una matriz de confusión cuenta etiquetas reales frente a etiquetas predichas ya disponibles. Para evitar ambigüedad, aquí las **filas son clases reales** y las **columnas son clases predichas**. La diagonal contiene aciertos; las otras celdas son confusiones.

No entrenamos un modelo en este bloque. La matriz solo resume las veinte predicciones simuladas del CSV y no mide un rendimiento general.

In [ ]:
orden_clases = ["normal", "alerta", "critica"]
matriz_confusion = pd.crosstab(predicciones["clase_real"], predicciones["clase_predicha"], rownames=["Clase real"], colnames=["Clase predicha"]).reindex(index=orden_clases, columns=orden_clases, fill_value=0)
display(matriz_confusion)

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(matriz_confusion, annot=True, fmt="d", cmap="Blues", cbar_kws={"label": "Número de predicciones"}, ax=ax)
ax.set(title="Matriz de confusión simulada")
plt.show()

## Práctica guiada

Crea un mapa de calor completo de `correlaciones` sin máscara. Usa anotaciones, paleta divergente, límites de -1 a 1 y barra de color. Después explica por qué el triángulo superior repite información y en qué caso preferirías no ocultarlo.

La salida debe tener cuatro filas, cuatro columnas y valores de diagonal iguales a 1.

In [ ]:
# Completa la práctica guiada aquí.


## Ejercicios

### Ejercicio 43.01 · Matriz comprobada

Calcula una correlación solo para las cuatro columnas numéricas. Comprueba que es cuadrada, simétrica y que la diagonal vale 1. Represéntala con una máscara triangular.

### Ejercicio 43.02 · Pivote con significado

Construye una tabla pivote de la media de `co2_ppm` por zona e intervalo de temperatura y represéntala. Indica en el título que son medias simuladas.

### Ejercicio 43.03 · Escala apropiada

Crea dos heatmaps de correlación: uno con escala automática y otro con `vmin=-1`, `vmax=1`, `center=0`. Explica por qué la segunda opción facilita comparar correlaciones entre matrices.

### Ejercicio 43.04 · Confusiones visibles

Calcula de nuevo `matriz_confusion`, verifica que suma 20 y crea un heatmap con filas reales y columnas predichas. Indica qué celda representa las observaciones críticas clasificadas como alerta.

### Ejercicio 43.05 · Miniproyecto: revisión de una tabla matricial

Elabora dos figuras: una matriz de correlaciones con diseño accesible y una matriz de confusión con etiquetas explícitas. Redacta tres conclusiones prudentes, incluyendo una asociación observada, una confusión concreta y un límite de estas muestras sintéticas.

In [ ]:
# Escribe aquí tu solución de los ejercicios 43.01 a 43.05.


## Autoevaluación y cierre

1. ¿Qué representa la diagonal de una matriz de correlación?
2. ¿Por qué una paleta divergente es útil para correlaciones?
3. ¿Qué decisión de agregación contiene una tabla pivote?
4. ¿Qué hace un `clustermap` y qué no demuestra?
5. ¿Qué representan filas y columnas de la matriz de confusión de esta unidad?

La siguiente unidad ampliará la exploración a paneles y a varias variables.